## **Archivo streamlit_app.py**

**BLOQUE 1: CARGA DE DATOS**

Lee el archivo df_clean.csv que ya tiene todas las columnas calculadas del notebook (duration_min, is_hit, tag, playcount, etc.). El @st.cache_data es un decorador de Streamlit que hace que el archivo se lea una sola vez — si el usuario interactúa con la app (mueve un slider, pulsa un botón), Streamlit no vuelve a leer el CSV desde disco, usa el que ya tiene en memoria. Sin esto la app sería lenta porque releería 34.000 filas en cada clic.

**BLOQUE 2: CARGA DEL MODELO**

Carga tres archivos que se generaron en el notebook:

1. modelo_hit.pkl — el Random Forest entrenado, el "cerebro" que hace las predicciones
2. encoder_tag.pkl — un diccionario que convierte géneros de texto a número ('pop' → 37, 'rock' → 42...). El modelo no entiende texto, solo números
3. features.pkl — una lista con los nombres de las 9 columnas en el orden exacto en que se entrenó el modelo. Este orden es crítico — si las columnas llegan en otro orden, las predicciones serían incorrectas

Se usa @st.cache_resource en lugar de @st.cache_data porque los modelos de sklearn son objetos grandes que no se deben copiar en memoria — cache_resource los comparte entre usuarios.

**BLOQUE 3: PREDICCIÓN**

Paso 1: El encoder convierte 'pop' en 37. Si el usuario escribe un género que el modelo nunca vio durante el entrenamiento, se pone 0 como valor por defecto.

Paso 2: Para una canción nueva no sabes cuántas reproducciones tendrá, pero el modelo las necesita porque fue entrenado con esa columna. La estimación es: oyentes × cuántas veces escucha cada uno = plays totales estimados. Si ese número supera los 5 millones (umbral p90 del dataset), is_hit_est = 1.

Paso 3: Se construye una tabla de una sola fila con todas las columnas. El .reindex(columns=features, fill_value=0) reordena las columnas en el orden exacto de features.pkl — si falta alguna columna la rellena con 0.

Paso 4: predict_proba() devuelve dos números: la probabilidad de que sea 0 (no hit) y la probabilidad de que sea 1 (hit). El [0][1] coge el segundo, que es el que nos interesa. Se multiplica por 100 para tenerlo en porcentaje.

**BLOQUE 4: GRÁFICOS**

Tres funciones, todas con el mismo patrón: reciben el DataFrame, crean una figura con matplotlib y la devuelven sin mostrarla. Streamlit la muestra después con st.pyplot().

1. plot_top_tracks — ordena por playcount descendente, toma los 15 primeros, hace un barplot horizontal. Divide por 1e6 para mostrar millones en lugar de números de 9 cifras.
2. plot_genres — agrupa por tag (género), suma los playcounts de todos los tracks de ese género, y ordena. Divide por 1e9 para mostrar en miles de millones.
3. plot_heatmap — filtra primero los tracks con país real (quita UNKNOWN y GLOBAL), luego construye una tabla pivote con géneros en las filas y países en las columnas, donde cada celda es la media de reproducciones. Seaborn pinta esa tabla como un mapa de calor donde los colores más oscuros = más reproducciones.

**BLOQUE 5: APP STREAMLIT**

Streamlit funciona de arriba a abajo — cada vez que el usuario interactúa, Python reejecutar todo el archivo desde el principio. Por eso el @st.cache_data y @st.cache_resource son importantes: evitan recargar el CSV y el modelo en cada reejercución.

Los st.columns() dividen la pantalla en columnas. El predictor usa dos columnas iguales [1, 1] — inputs a la izquierda, resultado a la derecha. Las pestañas (st.tabs) organizan los tres gráficos sin que la página sea infinitamente larga.

El if predecir: es clave — solo ejecuta la predicción cuando el usuario pulsa el botón. Sin ese if, la app intentaría predecir nada más cargar la página antes de que el usuario introduzca nada.